<a href="https://colab.research.google.com/github/jriveramerla/JRM-pySpark001/blob/master/JRM_Distributed_Processing_Challenges_Handling_Data_Skew_in_DF_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark= SparkSession.builder.appName('Practice').getOrCreate()

sc = spark.sparkContext

# Loading DataSkew
To understand skew, we create random data where keys are uniformly distributed

In [3]:
import numpy as np
import pandas as pd
import random


# sale dataset:
# table 1: OrderID, Qty, Sales, Discount (yes=1, no=0)
# table 2: ProductID, OrderID, Product, Price

########### Table 1 ###########

key_1 = [101] * 100
key_2 = [201] * 7000000
key_3 = [301] * 500
key_4 = [401] * 10000
OrderID = key_1 + key_2 + key_3 + key_4
random.shuffle(OrderID)

Qty_1 = list(np.random.randint(low = 1, high = 100, size = len(key_1)))
Qty_2 = list(np.random.randint(low = 1, high = 200, size = len(key_2)))
Qty_3 = list(np.random.randint(low = 1, high = 1000, size = len(key_3)))
Qty_4 = list(np.random.randint(low = 1, high = 50, size = len(key_4)))
Qty = Qty_1 + Qty_2 + Qty_3 + Qty_4

Sales_1 = list(np.random.randint(low = 10, high = 100, size = len(key_1)))
Sales_2 = list(np.random.randint(low = 50, high = 3400, size = len(key_2)))
Sales_3 = list(np.random.randint(low = 12, high = 2000, size = len(key_3)))
Sales_4 = list(np.random.randint(low = 40, high = 1000, size = len(key_4)))
Sales = Sales_1 + Sales_2 + Sales_3 + Sales_4

Discount = list(np.random.randint(low = 0, high = 2, size = len(OrderID)))
data1 = list(zip(OrderID,Qty,Sales,Discount))

# Create the Pandas DF
data_skew = pd.DataFrame(data1, columns=['OrderID','Qty','Sales','Discount'])


########### Table 2 ###########
data2 = [[1, 101, 'pencil', 4.99],
         [2, 101, 'book', 9.5],
         [3, 101, 'scissors', 14],
         [4, 301, 'glue', 7],
         [5, 201, 'marker', 8.49],
         [6, 301, 'label', 2],
         [7, 201, 'calculator', 3.99],
         [8, 501, 'eraser', 1.55],
        ]

data_small = pd.DataFrame(data2, columns=['ProductID', 'OrderID', 'Product', 'Price'])

In [4]:
# Create PySpark DF from Pandas

# Optimize conversion between PySpark and Pandas DF: Enable arrow-based columnar data transfers
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

df_skew = spark.createDataFrame(data_skew)
df_skew.printSchema()
df_skew.show()
df_skew.rdd.getNumPartitions()

root
 |-- OrderID: long (nullable = true)
 |-- Qty: long (nullable = true)
 |-- Sales: long (nullable = true)
 |-- Discount: long (nullable = true)

+-------+---+-----+--------+
|OrderID|Qty|Sales|Discount|
+-------+---+-----+--------+
|    201| 59|   74|       0|
|    201| 50|   79|       0|
|    201| 82|   37|       1|
|    201| 35|   62|       1|
|    201| 13|   91|       1|
|    201| 60|   27|       0|
|    201| 36|   86|       1|
|    201| 15|   97|       0|
|    201| 47|   65|       0|
|    201| 32|   39|       0|
|    201| 46|   96|       1|
|    201| 15|   65|       1|
|    201| 68|   78|       0|
|    201|  1|   80|       0|
|    201| 97|   83|       0|
|    201| 47|   90|       0|
|    201| 30|   10|       1|
|    201| 88|   28|       0|
|    201| 69|   26|       0|
|    201| 82|   29|       0|
+-------+---+-----+--------+
only showing top 20 rows


702

In [5]:
df_small = spark.createDataFrame(data_small)
df_small.printSchema()
df_small.show()
df_small.rdd.getNumPartitions()

root
 |-- ProductID: long (nullable = true)
 |-- OrderID: long (nullable = true)
 |-- Product: string (nullable = true)
 |-- Price: double (nullable = true)

+---------+-------+----------+-----+
|ProductID|OrderID|   Product|Price|
+---------+-------+----------+-----+
|        1|    101|    pencil| 4.99|
|        2|    101|      book|  9.5|
|        3|    101|  scissors| 14.0|
|        4|    301|      glue|  7.0|
|        5|    201|    marker| 8.49|
|        6|    301|     label|  2.0|
|        7|    201|calculator| 3.99|
|        8|    501|    eraser| 1.55|
+---------+-------+----------+-----+



2

# (2) Run a shuffle Join() with small sized data¶


In [6]:
joined_df = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

joined_df.show(30)

print(joined_df.count())

+-------+---+-----+--------+---------+-------+----------+-----+
|OrderID|Qty|Sales|Discount|ProductID|OrderID|   Product|Price|
+-------+---+-----+--------+---------+-------+----------+-----+
|    201| 59|   74|       0|        7|    201|calculator| 3.99|
|    201| 59|   74|       0|        5|    201|    marker| 8.49|
|    201| 50|   79|       0|        7|    201|calculator| 3.99|
|    201| 50|   79|       0|        5|    201|    marker| 8.49|
|    201| 82|   37|       1|        7|    201|calculator| 3.99|
|    201| 82|   37|       1|        5|    201|    marker| 8.49|
|    201| 35|   62|       1|        7|    201|calculator| 3.99|
|    201| 35|   62|       1|        5|    201|    marker| 8.49|
|    201| 13|   91|       1|        7|    201|calculator| 3.99|
|    201| 13|   91|       1|        5|    201|    marker| 8.49|
|    201| 60|   27|       0|        7|    201|calculator| 3.99|
|    201| 60|   27|       0|        5|    201|    marker| 8.49|
|    201| 36|   86|       1|        7|  

In [8]:
# DF increases the partition number to 200 automatically when spark performs data shuffling (join, aggregation)
print(joined_df.rdd.getNumPartitions())
joined_df.limit(3).rdd.glom().collect()

702


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [9]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

# run join()
joined_df = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")
joined_df.show(30)



+-------+---+-----+--------+---------+-------+----------+-----+
|OrderID|Qty|Sales|Discount|ProductID|OrderID|   Product|Price|
+-------+---+-----+--------+---------+-------+----------+-----+
|    201| 59|   74|       0|        7|    201|calculator| 3.99|
|    201| 59|   74|       0|        5|    201|    marker| 8.49|
|    201| 50|   79|       0|        7|    201|calculator| 3.99|
|    201| 50|   79|       0|        5|    201|    marker| 8.49|
|    201| 82|   37|       1|        7|    201|calculator| 3.99|
|    201| 82|   37|       1|        5|    201|    marker| 8.49|
|    201| 35|   62|       1|        7|    201|calculator| 3.99|
|    201| 35|   62|       1|        5|    201|    marker| 8.49|
|    201| 13|   91|       1|        7|    201|calculator| 3.99|
|    201| 13|   91|       1|        5|    201|    marker| 8.49|
|    201| 60|   27|       0|        7|    201|calculator| 3.99|
|    201| 60|   27|       0|        5|    201|    marker| 8.49|
|    201| 36|   86|       1|        7|  

In [10]:
print(joined_df.rdd.getNumPartitions())
joined_df.limit(3).rdd.glom().collect()

702


[[Row(OrderID=201, Qty=59, Sales=74, Discount=0, ProductID=7, OrderID=201, Product='calculator', Price=3.99),
  Row(OrderID=201, Qty=59, Sales=74, Discount=0, ProductID=5, OrderID=201, Product='marker', Price=8.49),
  Row(OrderID=201, Qty=50, Sales=79, Discount=0, ProductID=7, OrderID=201, Product='calculator', Price=3.99)]]

In [11]:
# perform a join() and descriptive statistics on a skewed data
from pyspark.sql.functions import *


groups = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

+------------------+-----------------+----------+----------+
|        AVG(Sales)|       STD(Sales)|MIN(Sales)|MAX(Sales)|
+------------------+-----------------+----------+----------+
|1722.6627193189204|967.5716327356981|        10|      3399|
+------------------+-----------------+----------+----------+



## Mitigate data skewness: SALTING¶


In [12]:
from pyspark.sql.functions import *

# add  random value [0 2]
df_skew_salting = df_skew.withColumn("_salt_", round(rand() * 2))
df_small_salting = df_small.withColumn("_salt_", round(rand() * 2))

df_skew_salting.show()
df_small_salting.show()

+-------+---+-----+--------+------+
|OrderID|Qty|Sales|Discount|_salt_|
+-------+---+-----+--------+------+
|    201| 59|   74|       0|   1.0|
|    201| 50|   79|       0|   1.0|
|    201| 82|   37|       1|   2.0|
|    201| 35|   62|       1|   1.0|
|    201| 13|   91|       1|   0.0|
|    201| 60|   27|       0|   2.0|
|    201| 36|   86|       1|   1.0|
|    201| 15|   97|       0|   1.0|
|    201| 47|   65|       0|   1.0|
|    201| 32|   39|       0|   0.0|
|    201| 46|   96|       1|   2.0|
|    201| 15|   65|       1|   0.0|
|    201| 68|   78|       0|   0.0|
|    201|  1|   80|       0|   0.0|
|    201| 97|   83|       0|   0.0|
|    201| 47|   90|       0|   0.0|
|    201| 30|   10|       1|   0.0|
|    201| 88|   28|       0|   2.0|
|    201| 69|   26|       0|   1.0|
|    201| 82|   29|       0|   2.0|
+-------+---+-----+--------+------+
only showing top 20 rows
+---------+-------+----------+-----+------+
|ProductID|OrderID|   Product|Price|_salt_|
+---------+-------+----

In [13]:
# repartition using _salt_:
df_skew_salting = df_skew_salting.repartition(3, "_salt_")
df_small_salting = df_small_salting.repartition(3, "_salt_")

In [14]:
df_skew_salting.limit(3).rdd.glom().collect()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [15]:
df_small_salting.rdd.glom().collect()


[[Row(ProductID=1, OrderID=101, Product='pencil', Price=4.99, _salt_=2.0),
  Row(ProductID=4, OrderID=301, Product='glue', Price=7.0, _salt_=2.0),
  Row(ProductID=7, OrderID=201, Product='calculator', Price=3.99, _salt_=2.0)],
 [Row(ProductID=3, OrderID=101, Product='scissors', Price=14.0, _salt_=0.0)],
 [Row(ProductID=2, OrderID=101, Product='book', Price=9.5, _salt_=1.0),
  Row(ProductID=5, OrderID=201, Product='marker', Price=8.49, _salt_=1.0),
  Row(ProductID=6, OrderID=301, Product='label', Price=2.0, _salt_=1.0),
  Row(ProductID=8, OrderID=501, Product='eraser', Price=1.55, _salt_=1.0)]]

In [16]:
# apply join()

df_skew_salting.drop("_salt_")
df_small_salting.drop("_salt_")


groups = df_skew_salting.join(df_small_salting, df_skew_salting.OrderID == df_small_salting.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

+------------------+-----------------+----------+----------+
|        AVG(Sales)|       STD(Sales)|MIN(Sales)|MAX(Sales)|
+------------------+-----------------+----------+----------+
|1722.6627193189204|967.5716327357006|        10|      3399|
+------------------+-----------------+----------+----------+



## (3) Run a shuffle Join() to see how the skew effects computation resources.¶


In [17]:
from pyspark.sql.functions import *

# set the number of partitions after shuffling (avoid 200 partitions)
spark.conf.set("spark.sql.shuffle.partitions", 8)

groups = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

+------------------+-----------------+----------+----------+
|        AVG(Sales)|       STD(Sales)|MIN(Sales)|MAX(Sales)|
+------------------+-----------------+----------+----------+
|1722.6627193189204|967.5716327356981|        10|      3399|
+------------------+-----------------+----------+----------+



## Mitigate data skewness: SALTING¶


In [18]:
from pyspark.sql.functions import *

# set the number of partitions after shuffling (avoid 200 partitions)
spark.conf.set("spark.sql.shuffle.partitions", 8)


# add  random value [0 7]
df_skew_salting = df_skew.withColumn("_salt_", round(rand() * 7))
df_small_salting = df_small.withColumn("_salt_", round(rand() * 7))

# repartition using _salt_:
df_skew_salting = df_skew_salting.repartition(8, "_salt_")
df_small_salting = df_small_salting.repartition(8, "_salt_")


# drop salting
df_skew_salting.drop("_salt_")
df_small_salting.drop("_salt_")


# apply join()
groups = df_skew_salting.join(df_small_salting, df_skew_salting.OrderID == df_small_salting.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

+------------------+----------------+----------+----------+
|        AVG(Sales)|      STD(Sales)|MIN(Sales)|MAX(Sales)|
+------------------+----------------+----------+----------+
|1722.6627193189204|967.571632735698|        10|      3399|
+------------------+----------------+----------+----------+



## Mitigate data skewness: Broadcast Hash Join¶


In [19]:
from pyspark.sql.functions import *

groups_brd = df_skew.join(broadcast(df_small), df_skew.OrderID == df_small.OrderID, how = "inner")

summary = groups_brd.select(mean(groups_brd.Sales).alias("AVG(Sales)"),
                        stddev(groups_brd.Sales).alias("STD(Sales)"),
                        min(groups_brd.Sales).alias("MIN(Sales)"),
                        max(groups_brd.Sales).alias("MAX(Sales)"),
                       )
summary.show()

+------------------+-----------------+----------+----------+
|        AVG(Sales)|       STD(Sales)|MIN(Sales)|MAX(Sales)|
+------------------+-----------------+----------+----------+
|1722.6627193189204|967.5716327356981|        10|      3399|
+------------------+-----------------+----------+----------+

